# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. `1 1.3333333333333333` — в Series один пропуск (`None` превращается в NaN), `isna().sum()` = 1. После `fillna(0)` значения 1, 0, 3, среднее 4/3.
2. `2` — после `strip` и `lower` значения превращаются в `"a"`, `"a"`, `"b"`, уникальных 2.
3. `3` — `drop_duplicates()` возвращает новую таблицу, а `df` не меняет. Результат нигде не сохранён, поэтому `len(df)` остаётся 3.
4. `1005` — после `replace(" ", "")` строки превращаются в `"1000"` и `"5"`, `to_numeric` переводит их в числа, сумма 1005.
5. `[9, 10]` — `.dt.month` берёт месяц из даты: у первой даты 9 (сентябрь), у второй 10 (октябрь), хотя заказы разделяют всего 11 часов.

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [ ]:
# 1
import pandas as pd
import numpy as np
from IPython.display import Markdown, display

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

def to_num(s):
    """Суммы вида '15 990' -> число. Пробелы (в том числе неразрывные) убираем, запятую считаем десятичной."""
    t = (s.astype(str).str.replace("\xa0", "", regex=False).str.replace(" ", "", regex=False)
          .str.replace(",", ".", regex=False))
    return pd.to_numeric(t, errors="coerce")

def drop_pct(d):
    """Изменение выручки оплаченных заказов, октябрь к сентябрю, в %."""
    ok = d["status"] == "оплачен"
    month = pd.to_datetime(d["order_date"]).dt.month
    s = d.loc[ok & (month == 9), "amount"].sum()
    o = d.loc[ok & (month == 10), "amount"].sum()
    return (o - s) / s * 100

# каждый шаг очистки отдельной функцией: так их можно применять по одному (нужно в задаче 3)
def s_amount(d):
    d = d.copy(); d["amount"] = to_num(d["amount"]); return d
def s_city(d):
    d = d.copy(); d["city"] = d["city"].astype(str).str.strip().str.lower().map(CITY); return d
def s_dedup(d):
    return d.drop_duplicates(subset="order_id", keep="first")        # оставляем первое вхождение order_id
def s_dates(d):
    d = d.copy(); d["order_date"] = pd.to_datetime(d["order_date"]); return d
def s_test(d):
    return d[d["customer_id"] != "TEST"]                              # отменённые НЕ удаляем

STEPS = [("суммы в числа", s_amount), ("города через словарь", s_city), ("дубли по order_id", s_dedup),
         ("даты в datetime", s_dates), ("без клиента TEST", s_test)]

def clean(path, verbose=True):
    d = pd.read_csv(path)
    log = [("исходные данные", len(d), 0, None)]
    if verbose: print(f"исходные данные: {len(d)} строк")
    for name, f in STEPS:
        before = len(d)
        d = f(d)
        pct = drop_pct(d)
        log.append((name, len(d), before - len(d), pct))
        if verbose: print(f"{name}: осталось {len(d)} строк (убрано {before - len(d)})")
    # проверки
    assert d["amount"].notna().all(), f"в amount остались NaN: {d['amount'].isna().sum()}"
    assert d["city"].notna().all(), f"в city остались NaN (не нашлось в словаре): {d['city'].isna().sum()}"
    assert d["order_id"].is_unique, "order_id не уникален"
    d = d.reset_index(drop=True)
    d.attrs["log"] = log
    return d

df = clean("orders.csv")
print("\nСтрок в результате:", len(df), "| городов:", df["city"].nunique(), "|", sorted(df["city"].unique()))
print("Отменённых заказов оставлено:", (df["status"] == "отменён").sum())
print(f"Менеджеру: после очистки осталось {len(df)} строк и {df['city'].nunique()} городов; отменённые заказы сохранены, чтобы считать долю отмен.")

## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [ ]:
# 2
paid = df["status"] == "оплачен"
month = df["order_date"].dt.month

def sep_oct(mask):
    s = df.loc[mask & paid & (month == 9), "amount"].sum()
    o = df.loc[mask & paid & (month == 10), "amount"].sum()
    return s, o, (o - s) / s * 100

s_all, o_all, c_all = sep_oct(pd.Series(True, index=df.index))
print(f"Всего: сентябрь {s_all:,.0f} ₸ -> октябрь {o_all:,.0f} ₸ ({c_all:+.1f}%)")
print("В условии ДЗ после полной очистки падение около -2.3%; у вас:", f"{c_all:+.1f}%")

rows = []
print(f"\n{'Город':<12}{'сентябрь':>14}{'октябрь':>14}{'Изм., %':>10}")
for c in sorted(df["city"].unique()):
    s, o, ch = sep_oct(df["city"] == c)
    rows.append((c, s, o, ch))
    print(f"{c:<12}{s:>14,.0f}{o:>14,.0f}{ch:>+10.1f}")
print("Сумма по городам равна общей выручке:",
      np.isclose(sum(r[1] for r in rows), s_all) and np.isclose(sum(r[2] for r in rows), o_all))

worst = min(rows, key=lambda r: r[3])
print("Сверьте строки этой таблицы с таблицей «город × месяц» урока 4: числа должны совпасть, если там использованы те же шаги очистки.")
print(f"Менеджеру: выручка за месяц изменилась на {c_all:+.1f}% ({s_all:,.0f} -> {o_all:,.0f} ₸). "
      f"Сильнее всех падает {worst[0]}: {worst[3]:+.1f}%, остальные города меняются в пределах "
      f"{min(r[3] for r in rows if r[0] != worst[0]):+.1f}…{max(r[3] for r in rows if r[0] != worst[0]):+.1f}%.")

## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [ ]:
# 3
log = df.attrs["log"]
base = s_amount(pd.read_csv("orders.csv"))             # исходные данные, но суммы уже числа (иначе выручку не посчитать)
base_pct = drop_pct(base)

lines = ["| Шаг очистки | Убрано строк | Падение выручки после шага | Сдвиг, п.п. | Если применить только этот шаг |",
         "|---|---:|---:|---:|---:|",
         f"| исходные данные (суммы переведены в числа) | 0 | {base_pct:+.1f}% | | |"]
prev, shifts = base_pct, []
for (name, rows_left, removed, pct), (_, f) in zip(log[1:], STEPS):
    if name == "суммы в числа":
        continue
    alone = drop_pct(f(base))
    shift = pct - prev
    shifts.append((name, removed, prev, pct, shift))
    lines.append(f"| {name} | {removed} | {pct:+.1f}% | {shift:+.1f} | {alone:+.1f}% |")
    prev = pct
display(Markdown("\n".join(lines)))

top = max(shifts, key=lambda t: abs(t[4]))
print(f"Итог: падение {base_pct:+.1f}% -> {prev:+.1f}% после очистки; итоговое число совпадает с задачей 2:", np.isclose(prev, c_all))
display(Markdown(f"**Менеджеру:** сильнее всего его цифру исказил шаг «{top[0]}»: он убрал {top[1]} строк и сдвинул падение выручки "
                 f"с {top[2]:+.1f}% до {top[3]:+.1f}% ({top[4]:+.1f} п.п.). Остальные шаги сдвигают его меньше."))

## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [ ]:
# 4
na = df["delivery_days"].isna()
print(f"Пропусков в delivery_days: {na.sum()} из {len(df)} ({na.mean():.1%})")

print("\nПо городам (пропусков и доля внутри города):")
for c in sorted(df["city"].unique()):
    m = df["city"] == c
    print(f"  {c:<12}{na[m].sum():>5}  {na[m].mean():>7.1%}")
print("\nПо статусам:")
for st in df["status"].unique():
    m = df["status"] == st
    print(f"  {st:<10}{na[m].sum():>5}  {na[m].mean():>7.1%}")
cancel_part = (df.loc[na, "status"] == "отменён").mean()
print(f"Среди пропусков отменённых заказов: {cancel_part:.1%}")

# Шымкент: доля доставок «за 2 дня». Считаем два варианта условия: не дольше 2 дней и ровно 2 дня
x = df.loc[df["city"] == "Шымкент", "delivery_days"]
med = x.median()
variants = {"(а) dropna": x.dropna(), f"(б) fillna(медиана={med:.0f})": x.fillna(med), "(в) fillna(0)": x.fillna(0)}
print(f"\nШымкент: {len(x)} заказов, из них без срока {x.isna().sum()}")
print(f"{'Вариант':<26}{'≤ 2 дней':>10}{'ровно 2':>10}{'знаменатель':>13}")
res = {}
for name, v in variants.items():
    res[name] = ((v <= 2).mean(), (v == 2).mean())
    print(f"{name:<26}{res[name][0]:>10.1%}{res[name][1]:>10.1%}{len(v):>13}")

a, b, c0 = list(res.values())
print(f"""Менеджеру: честный вариант (а) dropna: {a[0]:.1%} заказов в Шымкенте доставлены не дольше чем за 2 дня среди заказов, где срок известен.
(б) fillna(медиана) выдумывает {x.isna().sum()} сроков одинаковыми значениями ({b[0]:.1%}), (в) fillna(0) считает не доставленные заказы доставленными мгновенно и завышает долю до {c0[0]:.1%}.
Пропуски в основном у отменённых заказов ({cancel_part:.0%}), то есть доставки не было, и подставлять числа в них нельзя.""")

## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [ ]:
# 5
# Ошибка 1: df.drop_duplicates() возвращает НОВУЮ таблицу, результат нигде не сохранён, дубли остаются
#           (и дубли нужно искать по order_id, а не по полному совпадению строки).
# Ошибка 2: .str.lower() только меняет регистр: не убирает пробелы и не объединяет «almaty» и «Алматы»;
#           нужен strip и словарь CITY.
raw = pd.read_csv("orders.csv")

ai = raw.copy()
ai.drop_duplicates()
ai["city"] = ai["city"].str.lower()
print("Код AI: городов", ai["city"].nunique(), "| строк", len(ai), "(строк столько же, сколько в исходной таблице:", len(ai) == len(raw), ")")

fixed = raw.drop_duplicates(subset="order_id", keep="first").copy()      # результат сохраняем
fixed["city"] = fixed["city"].str.strip().str.lower().map(CITY)           # убираем пробелы, приводим к одному написанию
print("Исправлено: городов", fixed["city"].nunique(), "| строк", len(fixed), "| городов не нашлось в словаре:", fixed["city"].isna().sum())
print("Для сравнения: дубли по полной строке убрали бы", raw.drop_duplicates().shape[0], "строк, по order_id —", len(fixed))
print(f"Менеджеру: в исходной версии дубли не убирались ({len(raw) - len(fixed)} лишних строк), а города остались в {ai['city'].nunique()} написаниях; "
      f"после исправления городов {fixed['city'].nunique()}, строк {len(fixed)}.")

## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [ ]:
# 6
EN = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
RU = {"Monday": "понедельник", "Tuesday": "вторник", "Wednesday": "среда", "Thursday": "четверг",
      "Friday": "пятница", "Saturday": "суббота", "Sunday": "воскресенье"}

cnt = df["order_date"].dt.day_name().value_counts().reindex(EN).fillna(0).astype(int)
# в периоде дней недели не поровну, поэтому смотрим ещё на заказы в среднем за один такой день
n_days = df["order_date"].dt.normalize().drop_duplicates().dt.day_name().value_counts().reindex(EN)
per_day = cnt / n_days

print(f"{'День':<14}{'заказов':>9}{'дней в периоде':>16}{'в среднем за день':>19}")
for d_ in EN:
    print(f"{RU[d_]:<14}{cnt[d_]:>9}{n_days[d_]:>16.0f}{per_day[d_]:>19.1f}")

lo, hi = cnt.idxmin(), cnt.idxmax()
less = (1 - cnt[lo] / cnt[hi]) * 100
lo2, hi2 = per_day.idxmin(), per_day.idxmax()
print(f"\nМеньше всего заказов: {RU[lo]} ({cnt[lo]}), больше всего: {RU[hi]} ({cnt[hi]}); меньше на {less:.1f}%")
print(f"В пересчёте на один день: меньше всего {RU[lo2]} ({per_day[lo2]:.1f}), больше всего {RU[hi2]} ({per_day[hi2]:.1f}), разница {(1 - per_day[lo2]/per_day[hi2])*100:.1f}%")
print(f"Менеджеру: в {RU[lo]} заказов на {less:.0f}% меньше, чем в {RU[hi]} (в таблице учтены все статусы, включая отменённые). "
      f"Курьеров стоит ставить больше в {RU[hi]} и меньше в {RU[lo]}, но разницу в {less:.0f}% нужно проверить на более длинном периоде: данные всего за два месяца.")

## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [ ]:
# 7
sh = df[(df["city"] == "Шымкент") & df["delivery_days"].notna()].sort_values("order_date")
mon = sh["order_date"].dt.month
m_sep, m_oct = sh.loc[mon == 9, "delivery_days"].mean(), sh.loc[mon == 10, "delivery_days"].mean()
print(f"Средний срок доставки в Шымкенте: сентябрь {m_sep:.2f} дн., октябрь {m_oct:.2f} дн. (в {m_oct/m_sep:.1f} раза дольше)")

# последний заказ с быстрой доставкой (< 5 дней); первая дата после него: дальше быстрых уже нет
fast = sh[sh["delivery_days"] < 5]
last_fast = fast["order_date"].max()
after = sh[sh["order_date"] > last_fast]
brk = after["order_date"].min()
print("Последний заказ со сроком < 5 дней:", last_fast)
print("Первая дата, с которой срок всегда >= 5 дней:", brk.date(), "| минимум после неё:", after["delivery_days"].min(), "дн. |", len(after), "заказов")

print(f"\n{'Город':<12}{'до, дн.':>10}{'после, дн.':>12}{'разница':>10}")
others = []
for c in sorted(df["city"].unique()):
    x = df[(df["city"] == c) & df["delivery_days"].notna()]
    b4, af = x.loc[x["order_date"] < brk.normalize(), "delivery_days"].mean(), x.loc[x["order_date"] >= brk.normalize(), "delivery_days"].mean()
    print(f"{c:<12}{b4:>10.2f}{af:>12.2f}{af - b4:>+10.2f}")
    if c != "Шымкент":
        others.append(abs(af - b4))
    else:
        sh_b4, sh_af = b4, af
print("Максимальное изменение в других городах:", f"{max(others):.2f} дн.")

verdict = "в других городах срок почти не изменился" if max(others) < 0.5 else "в других городах тоже есть изменения, проверьте их"
print(f"""Менеджеру: с {brk.date()} срок доставки в Шымкенте вырос в среднем с {sh_b4:.1f} до {sh_af:.1f} дня и больше не бывает короче 5 дней; {verdict}.
Значит, проблема локальная (Шымкент), а не общая для всей сети. В первую очередь проверить, что менялось в Шымкенте в эту дату:
склад или пункт выдачи, курьерская служба, график работы, порядок передачи заказов.""")

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [ ]:
# 8
def to_amount(s):
    """Series строк -> числа. Понимает '15 990', '15990.0', '1 200,50'. Остальное ('нет данных') -> NaN.
    Правило: убираем пробелы (в т.ч. неразрывные), запятую считаем десятичной.
    Не рассчитано на формат с запятой как разделителем тысяч ('1,200.50')."""
    t = (s.astype(str).str.replace("\xa0", "", regex=False).str.replace(" ", "", regex=False)
          .str.replace(",", ".", regex=False))
    return pd.to_numeric(t, errors="coerce")             # errors="coerce": что не число, то NaN

test = pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])
res_t = to_amount(test)
print(res_t.tolist())
assert res_t.iloc[:3].tolist() == [15990.0, 15990.0, 1200.5] and np.isnan(res_t.iloc[3])

raw_amount = pd.read_csv("orders.csv")["amount"]
mine = to_amount(raw_amount)
lesson = pd.to_numeric(raw_amount.astype(str).str.replace(" ", "", regex=False), errors="coerce")   # способ из урока
print(f"Сумма to_amount: {mine.sum():,.2f} | способ урока: {lesson.sum():,.2f} | совпало: {np.isclose(mine.sum(), lesson.sum())}")
print("Значений, превращённых в NaN:", mine.isna().sum())
print("Менеджеру: функция понимает три формата записи суммы и не падает на «нет данных»; сумма по всему файлу совпадает с расчётом из урока.")